# サンプラー単体の処理時間 — Rustuna が効く条件を判定する

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 08 2026 年 9 月の現在地——Optuna v5.0 と Rustuna

**このノートのパス**　`ml-code/05_hpo/08_sampler_overhead.ipynb`

**このノートで確かめること**
**目的関数を空にして** `study.optimize` を回し、サンプラーの処理時間だけを取り出す。
**TPE は試行数とともに 1 試行あたりのコストが増える**ことを確かめ、
自分の探索での**サンプラー占有率**を計算して、高速実装が効くかを判定する。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `optuna numpy pandas` |
| 実行時間の目安 | 約 1〜2 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q optuna numpy pandas

### 背景

2026 年 9 月 7 日、Preferred Networks が Optuna の Rust 実装 **Rustuna** と **Optuna v5.0** を公開した。
公式は「ユースケースによっては Optuna に対し**最大 1,000 倍**高速に動作することを確認」
「探索時の消費メモリが**約 50% 削減**（※10 万試行時想定）」としている。

**これが自分に効くかどうかは、探索時間のうちサンプラーが何割かで決まる。**

In [3]:
import time
import numpy as np, pandas as pd, optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("optuna", optuna.__version__)


def sampler_overhead(sampler_cls, n_params, n_trials, seed=42):
    def objective(trial):                       # ← 学習しない。値を引くだけ
        return sum(trial.suggest_float(f"x{i}", -5, 5) for i in range(n_params))
    study = optuna.create_study(sampler=sampler_cls(seed=seed))
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=n_trials, n_jobs=1)
    return time.perf_counter() - t0

optuna 5.0.0


In [4]:
rows = []
for name, cls, trials_list in [
        ("TPESampler",    optuna.samplers.TPESampler,    [200, 1000]),
        ("RandomSampler", optuna.samplers.RandomSampler, [1000, 10000])]:
    for n_trials in trials_list:
        for n_params in [5, 20]:
            sec = sampler_overhead(cls, n_params, n_trials)
            rows.append({"sampler": name, "試行数": n_trials, "パラメータ数": n_params,
                         "秒": round(sec, 3),
                         "1試行あたり ms": round(sec / n_trials * 1000, 3)})
            print(rows[-1], flush=True)

overhead_df = pd.DataFrame(rows)
print()
print(overhead_df.to_string(index=False))

{'sampler': 'TPESampler', '試行数': 200, 'パラメータ数': 5, '秒': 0.434, '1試行あたり ms': 2.17}


{'sampler': 'TPESampler', '試行数': 200, 'パラメータ数': 20, '秒': 1.081, '1試行あたり ms': 5.405}


{'sampler': 'TPESampler', '試行数': 1000, 'パラメータ数': 5, '秒': 4.087, '1試行あたり ms': 4.087}


{'sampler': 'TPESampler', '試行数': 1000, 'パラメータ数': 20, '秒': 10.751, '1試行あたり ms': 10.751}


{'sampler': 'RandomSampler', '試行数': 1000, 'パラメータ数': 5, '秒': 0.429, '1試行あたり ms': 0.429}


{'sampler': 'RandomSampler', '試行数': 1000, 'パラメータ数': 20, '秒': 0.801, '1試行あたり ms': 0.801}


{'sampler': 'RandomSampler', '試行数': 10000, 'パラメータ数': 5, '秒': 4.609, '1試行あたり ms': 0.461}


{'sampler': 'RandomSampler', '試行数': 10000, 'パラメータ数': 20, '秒': 7.824, '1試行あたり ms': 0.782}



      sampler   試行数  パラメータ数      秒  1試行あたり ms
   TPESampler   200       5  0.434      2.170
   TPESampler   200      20  1.081      5.405
   TPESampler  1000       5  4.087      4.087
   TPESampler  1000      20 10.751     10.751
RandomSampler  1000       5  0.429      0.429
RandomSampler  1000      20  0.801      0.801
RandomSampler 10000       5  4.609      0.461
RandomSampler 10000      20  7.824      0.782


## 占有率の計算 — 判断はこの 1 行で決まる

In [5]:
# 書籍の旧条件（subsample_freq=0）では Optuna TPE が 40 試行で 154.0 秒だった
# 現在の 07_search_comparison.ipynb は subsample_freq=1。以下は必ず自分の実測値に置き換える
TOTAL_SEC, N_TRIALS = 154.0, 40           # ← 自分の探索の実測値に置き換える

trial_sec = TOTAL_SEC / N_TRIALS
tpe_ms = overhead_df.query(
    "sampler=='TPESampler' and 試行数==200 and パラメータ数==5")["1試行あたり ms"].iloc[0]

share = tpe_ms / 1000 / trial_sec
print(f"1 試行 = {trial_sec:.2f} 秒 / サンプラー = {tpe_ms:.2f} ミリ秒")
print(f"→ サンプラーの占有率 = {share*100:.3f} %")
print(f"→ 1,000 倍速くしたときの全体の短縮 = {share*100*(1-1/1000):.3f} %")
print()
if share < 0.01:
    print("判定: Rustuna は効かない。Pruning と並列化が先。")
elif share < 0.3:
    print("判定: 試行数を増やす計画があるなら Rustuna を検討する価値がある。")
else:
    print("判定: サンプラーが支配的。Rustuna の投入で全体が縮む。")

1 試行 = 3.85 秒 / サンプラー = 2.17 ミリ秒
→ サンプラーの占有率 = 0.056 %
→ 1,000 倍速くしたときの全体の短縮 = 0.056 %

判定: Rustuna は効かない。Pruning と並列化が先。


## 読み取りどころ

**TPE は 1 試行あたりのコストが試行数とともに増える。**5 次元で 200 試行なら約 3 ms、1,000 試行では約 6.6 ms と 2 倍以上になる。TPE は過去の全試行から分布を作り直すので、**履歴が伸びるほど 1 回が重くなる**。
対して `RandomSampler` は履歴を使わないので、1,000 試行でも 10,000 試行でもほぼ一定である。

**ここに Rustuna が生まれた理由がそのまま出ている。**PFN が「数十万回以上の大規模な探索」を対象に挙げているのは、この性質のため。

| サンプラーの占有率 | やるべきこと |
|---|---|
| **1% 未満** | Rustuna は効かない。**Pruning と並列化**が先。*ほとんどの機械学習タスクはここ。* |
| 数 %〜数十 % | 試行数を増やす計画があるなら検討する価値がある |
| **大半** | 試行が安く回数が莫大な領域。**Rustuna の投入で全体が縮む** |

⚠️ **「最大 1,000 倍」を自分のケースの見込みとして読まないこと。**公式の文言は「ユースケースによっては」「最大」である。

**一次資料**
- Preferred Networks「Rust による高速かつ省メモリな Optuna 実装『Rustuna』をリリース」(2026-09-07)
- Optuna 公式ドキュメント: `TPESampler`, `Study.optimize`